In [ ]:
# import requests
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

import time
import re
import pandas as pd
from pathlib import Path
from urllib.parse import parse_qs, urlparse

In [ ]:
EXCEL_NAME="posts_metas.xlsx"

SAVE_DIR = Path() / "datasets"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CONTENT_FOLDER_NAME = "woowa_content"
(SAVE_DIR / CONTENT_FOLDER_NAME).mkdir(parents=True, exist_ok=True)

In [ ]:
options = Options()

options.add_argument("--start-maximized")
url = "https://techblog.woowahan.com/"


def get_post_data_list(driver, wait):
  # 테스트용 포스트 제목 하나
  # post_list = wait.until(
  #   EC.presence_of_all_elements_located(
  #     (By.XPATH, "/html/body/div[4]/div[2]/div[2]/div[1]/div[1]/div[11]/a/h2")
  #   )
  # )

  # 포스트 리스트 div들 전부 가져오기
  post_list = wait.until(
    EC.presence_of_all_elements_located(
      (By.CSS_SELECTOR, "div.post-list > div.post-item:not(.firstpaint)")
    )
  )

  time.sleep(0.5)

  post_count = len(post_list)

  #  author | written_year | written_month | written_day | title | summary | link |
  post_data_list = []

  print(f"{post_count}개 포스팅 스크롤 시작")
  i = 1
  for post in post_list:
    print(f"\n --- {i}/{post_count} 번째 포스팅 --- \n")
    i+=1
    # 글쓴이
    # print(post.find_element(By.CSS_SELECTOR, "span.post-author-name").text)
    author = post.find_element(By.CSS_SELECTOR, "span.post-author-name").text

    # %Y %m %d
    match = re.search(r"(\d{4})\.\s*(\d{2})\.\s*(\d{2})\.", post.find_element(By.CSS_SELECTOR, "time.post-author-date").text)
    written_year = None
    written_month = None
    written_day = None
    if match:
      # print(f"{match.group(1)}, {match.group(2)}, {match.group(3)}")
      written_year = int(match.group(1))
      written_month = int(match.group(2))
      written_day = int(match.group(3))

    # 제목
    # print(post.find_element(By.CSS_SELECTOR, "h2.post-title").text)
    title = post.find_element(By.CSS_SELECTOR, "h2.post-title").text


    # 줄거리
    # print(post.find_element(By.CSS_SELECTOR, "p.post-excerpt").text)
    summary = post.find_element(By.CSS_SELECTOR, "p.post-excerpt").text

    # 링크
    # print(post.find_elements(By.CSS_SELECTOR, "a")[-1].get_attribute("href"))
    link = str(post.find_elements(By.CSS_SELECTOR, "a")[-1].get_attribute("href"))
    number = link.split("/")[-2]

    post_data_list.append({
      "author": author,
      "written_year": written_year,
      "written_month": written_month,
      "written_day": written_day,
      "title": title,
      "summary": summary,
      "link": link,
      "number": number
    })

  return post_data_list

  
# # # # # #
# 포스팅 구체적인 내용 가져오기
def get_post_link_list(driver, wait, post_data_list):
  post_data_detail_list = []
  for p in post_data_list:
    link = p.get("link")
    number = p.get("number")
    
    driver.get(link)

    time.sleep(1)
    post_data_detail_list.append({
      number: wait.until(
      EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "div.post-content-body:not(.pp-multiple-authors-boxes-wrapper)")
      )
    ).text})
  return post_data_detail_list


driver = webdriver.Chrome(options=options)
driver.get(url)

time.sleep(2)

wait = WebDriverWait(driver, 5)

wait.until(
  EC.element_to_be_clickable(
    (By.CSS_SELECTOR, "div.wp-pagenavi > a.last")
  )
).click()

# print("asdf")
# input("아무거나")

# 페이지 마지막부터 앞에까지 이동해주기

driver.refresh()
time.sleep(1)
url = driver.current_url
parse_url = urlparse(url)
query = parse_qs(parse_url.query)
last_page = int(query.get("paged", [None])[0])

post_data_list = []

for i in range(5, 0, -1):
  next_url = f"{parse_url.scheme}://{parse_url.netloc}?paged={i}"
  print(next_url)
  time.sleep(1)

  driver.get(next_url)
  driver.refresh()
  time.sleep(1)

  post_data_list.extend(get_post_data_list(driver, wait))

# # # # # #
# 포스팅들 메타데이터 xlsx로 뽑아보기
# print(post_data_list)
df = pd.DataFrame(post_data_list, columns=post_data_list[0].keys())
# df.head()

EXCEL_NAME = EXCEL_NAME.split(".")[0] + ".xlsx"

df.to_excel((SAVE_DIR / EXCEL_NAME), index=False)

if True:
  post_data_detail_list = get_post_link_list(driver, wait, post_data_list)

  for p_content in post_data_detail_list:
    print(p_content)
    content = list(p_content.values())[0]
    number = list(p_content.keys())[0]
    with open((SAVE_DIR / CONTENT_FOLDER_NAME / ("content_"+str(number)+".txt")), "w", encoding="utf-8-sig") as f:
      
      f.write(content)



https://techblog.woowahan.com?paged=5
10개 포스팅 스크롤 시작

 --- 1/10 번째 포스팅 --- 


 --- 2/10 번째 포스팅 --- 


 --- 3/10 번째 포스팅 --- 


 --- 4/10 번째 포스팅 --- 


 --- 5/10 번째 포스팅 --- 


 --- 6/10 번째 포스팅 --- 


 --- 7/10 번째 포스팅 --- 


 --- 8/10 번째 포스팅 --- 


 --- 9/10 번째 포스팅 --- 


 --- 10/10 번째 포스팅 --- 

https://techblog.woowahan.com?paged=4
10개 포스팅 스크롤 시작

 --- 1/10 번째 포스팅 --- 


 --- 2/10 번째 포스팅 --- 


 --- 3/10 번째 포스팅 --- 


 --- 4/10 번째 포스팅 --- 


 --- 5/10 번째 포스팅 --- 


 --- 6/10 번째 포스팅 --- 


 --- 7/10 번째 포스팅 --- 


 --- 8/10 번째 포스팅 --- 


 --- 9/10 번째 포스팅 --- 


 --- 10/10 번째 포스팅 --- 

https://techblog.woowahan.com?paged=3
10개 포스팅 스크롤 시작

 --- 1/10 번째 포스팅 --- 


 --- 2/10 번째 포스팅 --- 


 --- 3/10 번째 포스팅 --- 


 --- 4/10 번째 포스팅 --- 


 --- 5/10 번째 포스팅 --- 


 --- 6/10 번째 포스팅 --- 


 --- 7/10 번째 포스팅 --- 


 --- 8/10 번째 포스팅 --- 


 --- 9/10 번째 포스팅 --- 


 --- 10/10 번째 포스팅 --- 

https://techblog.woowahan.com?paged=2
10개 포스팅 스크롤 시작

 --- 1/10 번째 포스팅 --- 


 --- 2/10 번째 포스팅 --- 


 --- 3/10 번째 포스팅